# XY Probing and Skin-Stretch Analysis

This notebook owns the XY probing / skin-stretch tracking figures that were removed from the 2AFC psychophysics notebook. It reads psychophysics trial metadata, then analyzes the per-trial `pair_###/tracking.csv` files.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

CWD = Path.cwd()
if (CWD / "analysis" / "psychophysics_analysis" / "twoafc_psychophysics.py").exists():
    ROOT = CWD
elif (CWD.parent / "psychophysics_analysis" / "twoafc_psychophysics.py").exists():
    ROOT = CWD.parent.parent
else:
    ROOT = Path(__file__).resolve().parents[2] if "__file__" in globals() else CWD.parents[1]

PSYCH_DIR = ROOT / "analysis" / "psychophysics_analysis"
sys.path.insert(0, str(PSYCH_DIR))
import twoafc_psychophysics as pf

PSYCHOPHYSICS_OUTPUT_ROOT = PSYCH_DIR / "results"
OUTPUT_ROOT = ROOT / "analysis" / "probing_analysis" / "results"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
FIG_DPI = 160

print("Psychophysics outputs:", PSYCHOPHYSICS_OUTPUT_ROOT.resolve())
print("Probing outputs:", OUTPUT_ROOT.resolve())


## Load trial metadata

Run the psychophysics notebook through `combined_success_trials.csv` first if this file is missing.


In [ ]:
combined_success_trials_path = PSYCHOPHYSICS_OUTPUT_ROOT / "combined_success_trials.csv"
if not combined_success_trials_path.exists():
    raise FileNotFoundError(f"Missing {combined_success_trials_path}. Run the psychophysics notebook through the clean/success trials export first.")

combined_success_trials = pd.read_csv(combined_success_trials_path)
display(combined_success_trials.head())
print("combined success trials:", combined_success_trials.shape)


## Compute XY probing / skin-stretch tables and figures


In [ ]:
xy_probing_skin_stretch = pf.compute_xy_probing_and_skin_stretch_analysis(combined_success_trials)
xy_probing_skin_stretch_trial_summary = xy_probing_skin_stretch["trial_summary"]
xy_probing_skin_stretch_trajectory_bins = xy_probing_skin_stretch["trajectory_bins"]
xy_probing_skin_stretch_group_trajectory_bins = xy_probing_skin_stretch["group_trajectory_bins"]
xy_probing_skin_stretch_group_summary = xy_probing_skin_stretch["group_summary"]

pf.save_csv(xy_probing_skin_stretch_trial_summary, OUTPUT_ROOT, "xy_probing_skin_stretch_trial_summary.csv")
pf.save_csv(xy_probing_skin_stretch_trajectory_bins, OUTPUT_ROOT, "xy_probing_skin_stretch_trajectory_bins.csv")
pf.save_csv(xy_probing_skin_stretch_group_trajectory_bins, OUTPUT_ROOT, "xy_probing_skin_stretch_group_trajectory_bins.csv")
pf.save_csv(xy_probing_skin_stretch_group_summary, OUTPUT_ROOT, "xy_probing_skin_stretch_group_summary.csv")

xy_probing_skin_stretch_figure_paths = pf.save_xy_probing_skin_stretch_figures(
    OUTPUT_ROOT,
    xy_probing_skin_stretch_group_trajectory_bins,
    xy_probing_skin_stretch_trial_summary,
    xy_probing_skin_stretch_group_summary,
    FIG_DPI,
)

tracking_preview_columns = [
    "subject_id",
    "finger_condition",
    "trial_index_raw",
    "tracking_exists",
    "tracking_warning",
    "max_center_distance_px",
    "median_skin_stretch_gain_mm_per_m",
    "max_skin_stretch_command_proxy_px",
]
print(f"Saved {len(xy_probing_skin_stretch_figure_paths)} XY probing / skin-stretch figures")
display(xy_probing_skin_stretch_trial_summary[[c for c in tracking_preview_columns if c in xy_probing_skin_stretch_trial_summary.columns]].head(20))
display(pd.DataFrame({"figure": [str(p) for p in xy_probing_skin_stretch_figure_paths]}))
